In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from numpy.linalg import inv
import scipy.optimize as opt
from scipy import stats

# Question 1

In [ ]:
# Load data
annual_data = pd.read_excel('Assignment4Data.xlsx', sheet_name='ConsumptionAnnual')
quarterly_data = pd.read_excel('Assignment4Data.xlsx', sheet_name='ConsumptionQuarterly')

annual_data.head()

In [ ]:
# 1a)
# Load data
annual_data = pd.read_excel('Assignment4Data.xlsx', sheet_name='ConsumptionAnnual')
quarterly_data = pd.read_excel('Assignment4Data.xlsx', sheet_name='ConsumptionQuarterly')

# Prepare data
# Annual Data
annual_data['Log_Stock_Return'] = np.log((1 + annual_data['Stock'] / 100) / (1 + annual_data['Inflation'] / 100)) * 100
annual_data['Log_TBill_Return'] = np.log((1 + annual_data['T-Bill'] / 100) / (1 + annual_data['Inflation'] / 100)) * 100
annual_data['Excess_Return'] = annual_data['Log_Stock_Return'] - annual_data['Log_TBill_Return'].shift(1) 
annual_data['Log_Consumption'] = np.log(annual_data['C'])
annual_data['Log_Consumption_Growth'] = annual_data['Log_Consumption'].diff() * 100  
#annual_data = annual_data[annual_data['Year'] >= 1947] # Start from 47
annual_data.dropna(inplace=True)

# Quarterly Data
quarterly_data['Log_Stock_Return'] = np.log((1 + quarterly_data['Stock'] / 100) / (1 + quarterly_data['Inflation'] / 100)) * 100
quarterly_data['Log_TBill_Return'] = np.log((1 + quarterly_data['Bill'] / 100) / (1 + quarterly_data['Inflation'] / 100)) * 100
quarterly_data['Excess_Return'] = quarterly_data['Log_Stock_Return'] - quarterly_data['Log_TBill_Return'].shift(1)
quarterly_data['Log_Consumption'] = np.log(quarterly_data['C']) 
quarterly_data['Log_Consumption_Growth'] = quarterly_data['Log_Consumption'].diff() * 100
quarterly_data.dropna(inplace=True)

# Function for descriptive statistics
def compute_statistics_log(data, freq):
    stats = {}
    variables = ['Log_Consumption_Growth', 'Log_Stock_Return', 'Log_TBill_Return', 'Excess_Return']
    for var in variables:
        mean = data[var].mean()
        std = data[var].std()
        autocorr = data[var].autocorr(lag=1)

        # Annualize the quarterly data
        if freq == 'Quarterly':
            mean *= 4
            std *= 2

        stats[var] = {'Mean': mean, 'StdDev': std, 'Autocorr': autocorr}
    return pd.DataFrame(stats).T

# Annual data statistics
annual_stats_log = compute_statistics_log(annual_data, freq='Annual')
print("Annual Data Statistics (Log Returns):")
print(annual_stats_log)

# Quarterly data statistics
quarterly_stats_log = compute_statistics_log(quarterly_data, freq='Quarterly')
print("\nQuarterly Data Statistics (Log Returns):")
print(quarterly_stats_log)


In [ ]:
annual_data.head()

In [ ]:
# 1b)

# Moments - Annual
E_excess_return_annual_log = annual_data['Excess_Return'].mean()
var_stock_return_annual_log = annual_data['Log_Stock_Return'].var()
std_stock_return_annual_log = annual_data['Log_Stock_Return'].std()
std_consumption_growth_annual_log = annual_data['Log_Consumption_Growth'].std()
cov_excess_consump_annual_log = annual_data[['Log_Stock_Return', 'Log_Consumption_Growth']].cov().iloc[0, 1]
corr_excess_consump_annual_log = annual_data[['Log_Stock_Return', 'Log_Consumption_Growth']].corr().iloc[0, 1]

# Moments - Quarterly (but annualized)
E_excess_return_quarterly_log = quarterly_data['Excess_Return'].mean() * 4
var_stock_return_quarterly_log = quarterly_data['Log_Stock_Return'].var() * 4  # Annualize variance
std_stock_return_quarterly_log = quarterly_data['Log_Stock_Return'].std() * np.sqrt(4)
std_consumption_growth_quarterly_log = quarterly_data['Log_Consumption_Growth'].std() * np.sqrt(4)
cov_excess_consump_quarterly_log = quarterly_data[['Log_Stock_Return', 'Log_Consumption_Growth']].cov().iloc[0, 1] * 4 # should this be * 4?
corr_excess_consump_quarterly_log = quarterly_data[['Log_Stock_Return', 'Log_Consumption_Growth']].corr().iloc[0, 1]

print("\nAnnual Data Moments (Log Returns):")
print(f"Expected Excess Return: {E_excess_return_annual_log:.6f}")
print(f"Variance of Stock Return: {var_stock_return_annual_log:.6f}")
print(f"Std Dev of Stock Return: {std_stock_return_annual_log:.6f}")
print(f"Std Dev of Consumption Growth: {std_consumption_growth_annual_log:.6f}")
print(f"Covariance between Excess Return and Consumption Growth: {cov_excess_consump_annual_log:.6f}")
print(f"Correlation between Excess Return and Consumption Growth: {corr_excess_consump_annual_log:.6f}")

print("\nQuarterly Data Moments (Log Returns, Annualized):")
print(f"Expected Excess Return: {E_excess_return_quarterly_log:.6f}")
print(f"Variance of Stock Return: {var_stock_return_quarterly_log:.6f}")
print(f"Std Dev of Stock Return: {std_stock_return_quarterly_log:.6f}")
print(f"Std Dev of Consumption Growth: {std_consumption_growth_quarterly_log:.6f}")
print(f"Covariance between Excess Return and Consumption Growth: {cov_excess_consump_quarterly_log:.6f}")
print(f"Correlation between Excess Return and Consumption Growth: {corr_excess_consump_quarterly_log:.6f}")


In [ ]:
# 1c)

# Formula:

# Estimation of gamma with sample covariance 
# Annual
gamma_annual_cov_log = (E_excess_return_annual_log + var_stock_return_annual_log / 2) / cov_excess_consump_annual_log

# Quarterly
gamma_quarterly_cov_log = (E_excess_return_quarterly_log + var_stock_return_quarterly_log / 2) / cov_excess_consump_quarterly_log

# Estimation of gamma with correlation of 1 between excess returns and consumption growth
# Annual
sigma_ic_annual_corr1_log = std_stock_return_annual_log * std_consumption_growth_annual_log
gamma_annual_corr1_log = (E_excess_return_annual_log + var_stock_return_annual_log / 2) / sigma_ic_annual_corr1_log

# Quarterly
sigma_ic_quarterly_corr1_log = std_stock_return_quarterly_log * std_consumption_growth_quarterly_log
gamma_quarterly_corr1_log = (E_excess_return_quarterly_log + var_stock_return_quarterly_log / 2) / sigma_ic_quarterly_corr1_log

print("\nEstimated Risk Aversion coefficient gamma:")

print("\nUsing Sample Covariance (Annual Data):")
print(f"gamma = {gamma_annual_cov_log:.4f}")

print("Using Correlation = 1 (Annual Data):")
print(f"gamma = {gamma_annual_corr1_log:.4f}")

print("\nUsing Sample Covariance (Quarterly Data):")
print(f"gamma = {gamma_quarterly_cov_log:.4f}")

print("Using Correlation = 1 (Quarterly Data):")
print(f"gamma = {gamma_quarterly_corr1_log:.4f}")

In [ ]:
# 1d)

# We use the riskfree equation but solve for delta

# Compute consumption growth mean and variance
# Annual 
E_consumption_growth_annual = annual_data['Log_Consumption_Growth'].mean()
var_consumption_growth_annual = annual_data['Log_Consumption_Growth'].var()
print(E_consumption_growth_annual)
print(var_consumption_growth_annual)
print(gamma_annual_cov_log)

# Quarterly
E_consumption_growth_quarterly = quarterly_data['Log_Consumption_Growth'].mean() * 4
var_consumption_growth_quarterly = quarterly_data['Log_Consumption_Growth'].var() * 4

# Average riskfree rate
# Annual Data
rf_annual = annual_data['Log_TBill_Return'].mean()
print(rf_annual)

# Quarterly Data
rf_quarterly = quarterly_data['Log_TBill_Return'].mean() * 4

# Annual Data with gamma from covariance
delta_annual_cov = np.exp(-rf_annual + gamma_annual_cov_log * E_consumption_growth_annual - 0.5 * gamma_annual_cov_log ** 2 * var_consumption_growth_annual)
time_pref_rate_annual_cov = -np.log(delta_annual_cov)

# Annual Data with gamma from corr = 1
delta_annual_corr1 = np.exp(-rf_annual + gamma_annual_corr1_log * E_consumption_growth_annual - 0.5 * gamma_annual_corr1_log ** 2 * var_consumption_growth_annual)
time_pref_rate_annual_corr1 = -np.log(delta_annual_corr1)

# Quarterly Data with gamma from covariance
delta_quarterly_cov = np.exp(-rf_quarterly + gamma_quarterly_cov_log * E_consumption_growth_quarterly - 0.5 * gamma_quarterly_cov_log ** 2 * var_consumption_growth_quarterly)
time_pref_rate_quarterly_cov = -np.log(delta_quarterly_cov)

# Quarterly Data with gamma from corr = 1
delta_quarterly_corr1 = np.exp(-rf_quarterly + gamma_quarterly_corr1_log * E_consumption_growth_quarterly - 0.5 * gamma_quarterly_corr1_log ** 2 * var_consumption_growth_quarterly)
time_pref_rate_quarterly_corr1 = -np.log(delta_quarterly_corr1)


print("Estimated Time Discount Factor delta and Time Preference Rate (-ln(delta)):")

print("\nUsing Sample Covariance (Annual Data):")
print(f"delta = {delta_annual_cov:.6f}")
print(f"Time Preference Rate = {time_pref_rate_annual_cov:.6f}")

print("Using Correlation = 1 (Annual Data):")
print(f"delta = {delta_annual_corr1:.6f}")
print(f"Time Preference Rate = {time_pref_rate_annual_corr1:.6f}")

print("\nUsing Sample Covariance (Quarterly Data):")
print(f"delta = {delta_quarterly_cov:.6f}")
print(f"Time Preference Rate = {time_pref_rate_quarterly_cov:.6f}")

print("Using Correlation = 1 (Quarterly Data):")
print(f"delta = {delta_quarterly_corr1:.6f}")
print(f"Time Preference Rate = {time_pref_rate_quarterly_corr1:.6f}")



# Question 2

In [ ]:
# Use annual data
data = pd.read_excel('Assignment4Data.xlsx', sheet_name='ConsumptionAnnual')

# Data
data['Consumption_Growth'] = data['C'] / data['C'].shift(1)
data['Gross_Stock_Return'] = 1 + data['Stock'].shift(-1) / 100
data['Gross_TBill_Return'] = 1 + data['T-Bill'] / 100
data['Excess_Return_t1'] = (1 + (data['Stock'].shift(-1) / 100 - data['T-Bill']) / 100)

# Instruments
data['Instrument_1'] = 1
data['Instrument_2'] = data['Consumption_Growth'].shift(1)
data['Instrument_3'] = 1 + (data['Stock'] / 100 - data['T-Bill'].shift(1) / 100)

data.dropna(inplace = True)


data.head()

In [ ]:
# 2a)
# Consumption growth Ct+1 / Ct
consumption_growth = data['Consumption_Growth'].values

# Gross returns
Rm_t1 = data['Gross_Stock_Return'].values
Rf_t1 = data['Gross_TBill_Return'].values

# Instruments z_t
z_t = data[['Instrument_1', 'Instrument_2', 'Instrument_3']].values

# Initial parameters
delta_init = 1
gamma_init = 1
params_init = np.array([delta_init, gamma_init])

# Initial w matrix, identity
W = np.eye(6)

# GMM function
def gmm_objective(params):
    delta, gamma = params
    
    # Stochastic discount factor (SDF)
    SDF = delta * consumption_growth ** (-gamma) 
    
    # Moment conditions for stock returns
    m1_scalar = SDF * Rm_t1 - 1 
    m1 = m1_scalar[:, np.newaxis] * z_t  
    
    # Moment conditions for T-bill returns
    m2_scalar = SDF * Rf_t1 - 1 
    m2 = m2_scalar[:, np.newaxis] * z_t 
    
    # Stack moments
    moments = np.hstack((m1, m2))  # Shape (n_samples, 6)
    
    # Calculate g matrix
    g = moments.mean(axis=0)  # Shape (6,)
    
    # Compute the GMM criterion J = g' W g
    J = g.T @ W @ g
    return J

# Below is the 2nd step GMM function but that uses an optimal W matrix (W_opt) which is calculated from the first step. 
def gmm_objective_opt(params):
    delta, gamma = params
    
    # Stochastic discount factor (SDF)
    SDF = delta * consumption_growth ** (-gamma)
    
    # Moment conditions for stock returns
    m1_scalar = SDF * Rm_t1 - 1
    m1 = m1_scalar[:, np.newaxis] * z_t
    
    # Moment conditions for T-bill returns
    m2_scalar = SDF * Rf_t1 - 1
    m2 = m2_scalar[:, np.newaxis] * z_t
    
    # Stack moments
    moments = np.hstack((m1, m2))
    
    # Calculate g matrix
    g = moments.mean(axis=0)
    
    # Compute the GMM criterion J = g' W_opt g
    J = g.T @ W_opt @ g  # With w_opt from first step
    return J

# Run first step GMM with initial guesses
result = opt.minimize(gmm_objective, params_init, method='Nelder-Mead')
params_estimated = result.x
delta_est, gamma_est = params_estimated


# Compute SDF
SDF_est = delta_est * consumption_growth ** (-gamma_est)

# Moments with SDF
# Moments for stock returns
m1_scalar_est = SDF_est * Rm_t1 - 1
m1_est = m1_scalar_est[:, np.newaxis] * z_t

# # Moments for T-bill returns
m2_scalar_est = SDF_est * Rf_t1 - 1
m2_est = m2_scalar_est[:, np.newaxis] * z_t

# # Stack moment conditions
moments_est = np.hstack((m1_est, m2_est))

# # Optimal weighting matrix with one lag

moments_flat = moments_est.reshape(moments_est.shape[0], -1)

def hac_covariance(moments, nlags=1):
    T, K = moments.shape  # T = number of observations, K = number of variables
    mean_moment = moments.mean(axis=0)  # Mean of each column
    demeaned = moments - mean_moment  # Demeaned matrix
    S = np.cov(demeaned, rowvar=False)  # Variance-covariance matrix at lag 0

    for lag in range(1, nlags + 1):
        weight = 1 - lag / (nlags + 1)  # bartlett kernel weight
        gamma = (demeaned[:-lag].T @ demeaned[lag:]) / T  # Autocovariance at lag
        S += weight * (gamma + gamma.T)  # Add autocovariance with weight

    return S

# NW covariance 
S_hac = hac_covariance(moments_flat, nlags=1)
W_opt = inv(S_hac)

# Second step GMM
result_opt = opt.minimize(gmm_objective_opt, params_estimated, method='Nelder-Mead')
params_final = result_opt.x
delta_final, gamma_final = params_final




In [ ]:
# Compute the jacobian numerically for SEs

epsilon = 1e-6  # Small change
num_params = 2  # delta nad gamma parameters
num_moments = 6  # Number of moment conditions
D = np.zeros((num_moments, num_params))

# Parameters at the estimated values
delta_hat = delta_final
gamma_hat = gamma_final

# Loop over parameters to compute numerical derivatives
for i in range(num_params):
    # Big and small parameters
    params_up = params_final.copy()
    params_down = params_final.copy()
    params_up[i] += epsilon
    params_down[i] -= epsilon
    
    # Moments at params_up
    delta_up, gamma_up = params_up
    SDF_up = delta_up * consumption_growth ** (-gamma_up)
    m1_scalar_up = SDF_up * Rm_t1 - 1
    m1_up = m1_scalar_up[:, np.newaxis] * z_t
    m2_scalar_up = SDF_up * Rf_t1 - 1
    m2_up = m2_scalar_up[:, np.newaxis] * z_t
    moments_up = np.hstack((m1_up, m2_up))
    g_up = moments_up.mean(axis=0)
    
    # Moments at params_down
    delta_down, gamma_down = params_down
    SDF_down = delta_down * consumption_growth ** (-gamma_down)
    m1_scalar_down = SDF_down * Rm_t1 - 1
    m1_down = m1_scalar_down[:, np.newaxis] * z_t
    m2_scalar_down = SDF_down * Rf_t1 - 1
    m2_down = m2_scalar_down[:, np.newaxis] * z_t
    moments_down = np.hstack((m1_down, m2_down))
    g_down = moments_down.mean(axis=0)
    
    # Numerical derivative
    D[:, i] = (g_up - g_down) / (2 * epsilon)

# Variance covariance and SE
# Asymptotic variance of parameters
Var_params = inv(D.T @ W_opt @ D)
std_errors = np.sqrt(np.diag(Var_params))

# Mean moments
# Moments at estimated parameters
SDF_hat = delta_hat * consumption_growth ** (-gamma_hat)
m1_scalar_hat = SDF_hat * Rm_t1 - 1
m1_hat = m1_scalar_hat[:, np.newaxis] * z_t
m2_scalar_hat = SDF_hat * Rf_t1 - 1
m2_hat = m2_scalar_hat[:, np.newaxis] * z_t
moments_hat = np.hstack((m1_hat, m2_hat))
g_hat = moments_hat.mean(axis=0)

# J-stat
n = len(data) 
J_stat = n * (g_hat.T @ W_opt @ g_hat)
# P values
degrees_of_freedom = num_moments - num_params  # 6 moments - 2 parameters
p_value = 1 - stats.chi2.cdf(J_stat, degrees_of_freedom)

print("GMM Estimation Results (Two-Step Estimator):")
print(f"Estimated δ: {delta_final:.6f} (Std Error: {std_errors[0]:.6f})")
print(f"Estimated γ: {gamma_final:.6f} (Std Error: {std_errors[1]:.6f})")
print(f"J-statistic: {J_stat:.4f}")
print(f"P-value of overidentifying restrictions test: {p_value:.4f}")



In [ ]:
# 2b)

# Grab data
# Consumption growth Ct+1 / Ct
consumption_growth = data['Consumption_Growth'].values 

# Excess return on stocks: Rm,t+1 - Rf,t
excess_return = data['Excess_Return_t1'].values 

# Instruments z_t
z_t = data[['Instrument_1', 'Instrument_2', 'Instrument_3']].values

# Intial guess
gamma_init = 1.0
params_init = np.array([gamma_init])
W = np.eye(3)  # 3 moment conditions

# GMM Setup
def gmm_objective(params):
    gamma = params[0]
    
    # Compute the stochastic discount factor (SDF)
    SDF = consumption_growth ** (-gamma)  # Shape (n_samples,)
    
    # Compute the moment conditions
    m_scalar = SDF * excess_return  # Shape (n_samples,)
    moments = m_scalar[:, np.newaxis] * z_t  # Shape (n_samples, 3)
    
    # Compute the average moment conditions (g)
    g = moments.mean(axis=0)  # Shape (3,)
    
    # Compute the GMM criterion J = g' W g
    J = g.T @ W @ g
    return J

# First step opt
result = opt.minimize(gmm_objective, params_init, method='Nelder-Mead')
gamma_estimated = result.x[0]


# Compute SDF and moments
SDF_est = consumption_growth ** (-gamma_estimated)
m_scalar_est = SDF_est * excess_return
moments_est = m_scalar_est[:, np.newaxis] * z_t  # Shape (n_samples, 3)

# Optimal weighting matrix
# Compute the HAC covariance matrix
moments_flat = moments_est.reshape(moments_est.shape[0], -1) # flat for HAC function
S_hac = hac_covariance(moments_flat, nlags=1)
W_opt = inv(S_hac)


# Second step
def gmm_objective_opt(params):
    gamma = params[0]
    
    # Compute the stochastic discount factor (SDF)
    SDF = consumption_growth ** (-gamma)
    
    # Compute the moment conditions
    m_scalar = SDF * excess_return
    moments = m_scalar[:, np.newaxis] * z_t
    
    # Compute the average moment conditions (g)
    g = moments.mean(axis=0)
    
    # Compute the GMM criterion J = g' W_opt g
    J = g.T @ W_opt @ g
    return J

result_opt = opt.minimize(gmm_objective_opt, [gamma_estimated], method='Nelder-Mead')
gamma_final = result_opt.x[0]



In [ ]:
# Compute Jacobian
epsilon = 1e-6  # Small change
num_params = 1  # Gamma
num_moments = 3  # Number of moment conditions
D = np.zeros((num_moments, num_params))

# Parameter at the estimated value
gamma_hat = gamma_final

# Bigger and smaller parameter
params_up = np.array([gamma_hat + epsilon])
params_down = np.array([gamma_hat - epsilon])

# Moments at params_up
gamma_up = params_up[0]
SDF_up = consumption_growth ** (-gamma_up)
m_scalar_up = SDF_up * excess_return
moments_up = m_scalar_up[:, np.newaxis] * z_t
g_up = moments_up.mean(axis=0)

# Moments at params_down
gamma_down = params_down[0]
SDF_down = consumption_growth ** (-gamma_down)
m_scalar_down = SDF_down * excess_return
moments_down = m_scalar_down[:, np.newaxis] * z_t
g_down = moments_down.mean(axis=0)

# Numerical derivative
D[:, 0] = (g_up - g_down) / (2 * epsilon)

# Variance Covariance and SE
# Asymptotic variance
Var_params = inv(D.T @ W_opt @ D)

std_error = np.sqrt(Var_params[0, 0])

# Moments at estimated gamma
SDF_hat = consumption_growth ** (-gamma_hat)
m_scalar_hat = SDF_hat * excess_return
moments_hat = m_scalar_hat[:, np.newaxis] * z_t
g_hat = moments_hat.mean(axis=0)

# Jstat and pvalues
n = len(data)  # Number of observations
J_stat = n * (g_hat.T @ W_opt @ g_hat)
degrees_of_freedom = num_moments - num_params  # 3 moments - 1 parameter
p_value = 1 - stats.chi2.cdf(J_stat, degrees_of_freedom)

print("GMM Estimation Results (Two-Step Estimator - Excess Returns):")
print(f"Estimated γ: {gamma_final:.6f} (Std Error: {std_error:.6f})")
print(f"J-statistic: {J_stat:.4f}")
print(f"P-value of overidentifying restrictions test: {p_value:.4f}")


In [ ]:
#2b Iterative estimator

# Options
max_iter = 10
tol = 1e-6
gamma_iter = gamma_estimated  # Start from initial estimate

for iteration in range(max_iter):
    # Compute SDF at current gamma
    SDF_iter = consumption_growth ** (-gamma_iter)
    
    # Compute moment conditions
    m_scalar_iter = SDF_iter * excess_return
    moments_iter = m_scalar_iter[:, np.newaxis] * z_t
    g_iter = moments_iter.mean(axis=0)
    
    # Compute covariance matrix with Newey-West estimator
    moments_flat = moments_iter.reshape(moments_iter.shape[0], -1)
    S_iter = hac_covariance(moments_flat, nlags=1)
    
    # Compute optimal weighting matrix
    W_iter = inv(S_iter)
    
    # Define GMM objective function with current weighting matrix
    def gmm_objective_iter(params):
        gamma = params[0]
        SDF = consumption_growth ** (-gamma)
        m_scalar = SDF * excess_return
        moments = m_scalar[:, np.newaxis] * z_t
        g = moments.mean(axis=0)
        J = g.T @ W_iter @ g
        return J
    
    # Perform optimization
    result_iter = opt.minimize(gmm_objective_iter, [gamma_iter], method='Nelder-Mead')
    gamma_new = result_iter.x[0]
    
    # Check convergence
    if abs(gamma_new - gamma_iter) < tol:
        print(f"Converged after {iteration+1} iterations.")
        break
    else:
        gamma_iter = gamma_new
else:
    print("Maximum iterations reached without convergence.")

# Moments at final gamma
SDF_final = consumption_growth ** (-gamma_iter)
m_scalar_final = SDF_final * excess_return
moments_final = m_scalar_final[:, np.newaxis] * z_t
g_final = moments_final.mean(axis=0)

# Covariance matrix and weighting matrix
moments_final_flat = moments_final.reshape(moments_final.shape[0], -1)
S_final = hac_covariance(moments_final_flat, nlags=1)
W_final = inv(S_final)

# Compute Jacobian matrix D at final gamma
params_up = np.array([gamma_iter + epsilon])
params_down = np.array([gamma_iter - epsilon])

# Moments at params_up
gamma_up = params_up[0]
SDF_up = consumption_growth ** (-gamma_up)
m_scalar_up = SDF_up * excess_return
moments_up = m_scalar_up[:, np.newaxis] * z_t
g_up = moments_up.mean(axis=0)

# Moments at params_down
gamma_down = params_down[0]
SDF_down = consumption_growth ** (-gamma_down)
m_scalar_down = SDF_down * excess_return
moments_down = m_scalar_down[:, np.newaxis] * z_t
g_down = moments_down.mean(axis=0)

# Numerical derivative
D_final = np.zeros((num_moments, num_params))
D_final[:, 0] = (g_up - g_down) / (2 * epsilon)

# Variance of gamma
Var_gamma_final = inv(D_final.T @ W_final @ D_final)
std_error_final = np.sqrt(Var_gamma_final[0, 0])

# Jstat and pvalue
J_stat_final = n * (g_final.T @ W_final @ g_final)
p_value_final = 1 - stats.chi2.cdf(J_stat_final, degrees_of_freedom)

print("\nGMM Estimation Results (Iterated Estimator - Excess Returns):")
print(f"Estimated γ: {gamma_iter:.6f} (Std Error: {std_error_final:.6f})")
print(f"J-statistic: {J_stat_final:.4f}")
print(f"P-value of overidentifying restrictions test: {p_value_final:.4f}")
